In [ ]:
# =====================================================================
# GENESIS NATALITY v3.0 — DEEP SIMULATION OPTIMIZER (ALL AGES × STAGES)
# =====================================================================
# For each (age,stage):
#   - Outer: Differential Evolution proposes doses
#   - Inner: Monte Carlo sim runs (N_sim) with noise/uncertainty
#   - Objective: minimize  -E[Score] + lambda * Std[Score]
# =====================================================================

import numpy as np
import pandas as pd
import warnings
import torch
from scipy.optimize import differential_evolution

warnings.filterwarnings("ignore")
np.random.seed(2041)
torch.manual_seed(2041)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 GENESIS NATALITY v3.0 ONLINE | DEVICE: {DEVICE}")

# ----------------------------- Vault ---------------------------------
class NatalityVault:
    def __init__(self):
        self.library = {
            "Methylfolate (B9)":   {"reference": 5.0,    "slope": 0.8, "system": "Neuro"},
            "Iron Bisglycinate":   {"reference": 60.0,   "slope": 0.6, "system": "Blood"},
            "Vitamin D3":          {"reference": 4000.0, "slope": 0.4, "system": "Immune"},
            "Omega-3 (DHA/EPA)":   {"reference": 2000.0, "slope": 0.7, "system": "Neuro"},
            "Choline (CDP)":       {"reference": 1000.0, "slope": 0.5, "system": "Neuro"},
            "Magnesium Threonate": {"reference": 400.0,  "slope": 0.3, "system": "Metabolic"},
            "Iodine":              {"reference": 250.0,  "slope": 0.4, "system": "Thyroid"},
            "Calcium":             {"reference": 1500.0, "slope": 0.3, "system": "Vascular"},
            "Multi-Trace (Zn/Se)": {"reference": 50.0,   "slope": 0.3, "system": "Immune"},
            "Low-Dose Aspirin":    {"reference": 81.0,   "slope": 0.3, "system": "Flow"},
            "Metformin (XR)":      {"reference": 1500.0, "slope": 0.5, "system": "Metabolic"},
            "Antihypertensive":    {"reference": 200.0,  "slope": 0.5, "system": "Vascular"},
            "Insulin (Basal)":     {"reference": 30.0,   "slope": 0.6, "system": "Metabolic"},
            "NMN":                 {"reference": 500.0,  "slope": 0.9, "system": "Mito"},
            "Resveratrol":         {"reference": 250.0,  "slope": 0.2, "system": "Mito"},
            "Spermidine":          {"reference": 10.0,   "slope": 0.7, "system": "Cell"},
            "PQQ":                 {"reference": 20.0,   "slope": 0.4, "system": "Mito"},
            "BPC-157":             {"reference": 0.5,    "slope": 1.2, "system": "Repair"},
            "AAV-BDNF":            {"reference": 1.0,    "slope": 2.5, "system": "Neuro++"},
            "Exosomes (MSC)":      {"reference": 1.0,    "slope": 1.5, "system": "Immune++"},
        }
        self.agents = list(self.library.keys())
        self.n_agents = len(self.agents)

        self.interactions = pd.DataFrame(0.0, index=self.agents, columns=self.agents)
        self._set_interaction("Methylfolate (B9)", "Choline (CDP)", 0.25)
        self._set_interaction("NMN", "Resveratrol", 0.35)
        self._set_interaction("Iron Bisglycinate", "Magnesium Threonate", -0.1)
        self._set_interaction("Metformin (XR)", "BPC-157", 0.1)

    def _set_interaction(self, a, b, v):
        self.interactions.loc[a, b] = v
        self.interactions.loc[b, a] = v

# --------------------------- Stages & Ages ----------------------------
STAGES = [
    "preconception",
    "fertilization_window",
    "implantation",
    "trimester_1",
    "trimester_2",
    "trimester_3",
    "postpartum",
]
AGE_BANDS = [20, 30, 40, 50]

class StageObjectives:
    def get_weights(self, stage, age):
        w_implant = w_growth = w_neuro = w_vasc = w_prog = 0.0
        if stage == "preconception":
            w_implant, w_neuro, w_vasc, w_prog = 0.4, 0.2, 0.2, 0.2
        elif stage == "fertilization_window":
            w_implant, w_vasc, w_prog = 0.6, 0.2, 0.2
        elif stage == "implantation":
            w_implant, w_vasc, w_prog = 0.5, 0.3, 0.2
        elif stage == "trimester_1":
            w_growth, w_neuro, w_vasc, w_prog = 0.2, 0.4, 0.2, 0.2
        elif stage == "trimester_2":
            w_growth, w_neuro, w_vasc, w_prog = 0.4, 0.3, 0.2, 0.1
        elif stage == "trimester_3":
            w_growth, w_vasc, w_prog = 0.5, 0.3, 0.2
        elif stage == "postpartum":
            w_growth, w_neuro, w_vasc, w_prog = 0.2, 0.3, 0.2, 0.3

        age_factor = np.clip((age - 20) / 30.0, 0.0, 1.0)
        w_implant *= 1.0 + 0.3 * age_factor
        w_vasc    *= 1.0 + 0.3 * age_factor
        w_prog    *= 1.0 + 0.2 * (1.0 - age_factor)

        total = w_implant + w_growth + w_neuro + w_vasc + w_prog
        if total == 0:
            total = 1.0
        return {
            "implant":  w_implant / total,
            "growth":   w_growth  / total,
            "neuro":    w_neuro   / total,
            "vascular": w_vasc    / total,
            "program":  w_prog    / total,
        }

# --------------------------- Physiology Engine ------------------------
class StagePhysiologyEngine:
    """
    Provides a single deterministic score for a given dose vector, stage, and age.
    The optimizer will wrap this with Monte Carlo noise.
    """

    def __init__(self, vault):
        self.vault = vault
        self.obj = StageObjectives()

    def _benefit(self, dose, ref, slope):
        rel = dose / (ref + 1e-8)
        return slope * 8.0 * (rel / (1.0 + rel))

    def _toxicity(self, dose, ref, stage, system):
        """
        Tightened toxicity specifically for fertilization/implantation,
        while keeping other stages aggressive.
        """
        rel = dose / (ref + 1e-8)
    
        # Base toxicity curve (same shape as before)
        if rel <= 1.0:
            base = 0.2 * rel
        else:
            base = 0.2 + 2.0 * (np.exp((rel - 1.0) * 1.0) - 1.0)
    
        # Stage-specific multipliers
        stage_mult = 1.0
    
        # STRONGER penalty only for very experimental systems during
        # fertilization + implantation to prevent negative scores there.
        if stage in ["fertilization_window", "implantation"]:
            if system in ["Neuro++", "Immune++", "Repair", "Mito", "Cell"]:
                stage_mult = 2.5   # was ~1.5 in the aggressive version
    
        # Mildly higher penalty in preconception for gene/immune++
        elif stage == "preconception":
            if system in ["Neuro++", "Immune++", "Repair"]:
                stage_mult = 1.5   # modest bump vs other stages
    
        # All other stages remain relatively permissive
        return base * stage_mult



    def score(self, doses, stage, age):
        w = self.obj.get_weights(stage, age)
        impl = growth = neuro = vasc = prog = 0.0
        tox_total = 0.0

        for i, agent in enumerate(self.vault.agents):
            d = doses[i]
            meta = self.vault.library[agent]
            ref = meta["reference"]
            slope = meta["slope"]
            system = meta["system"]

            ben = self._benefit(d, ref, slope)
            tox = self._toxicity(d, ref, stage, system)
            tox_total += tox

            if system in ["Blood", "Thyroid", "Metabolic"]:
                growth += ben
            if system in ["Neuro", "Neuro++"]:
                neuro += ben
            if system in ["Vascular", "Flow"]:
                vasc += ben
            if system in ["Mito", "Cell", "Repair", "Immune", "Immune++"]:
                prog += ben
            if system in ["Neuro", "Immune", "Vascular", "Thyroid"]:
                impl += 0.25 * ben

        synergy = 0.0
        active = np.where(doses > 0.01)[0]
        for i in active:
            for j in active:
                if j <= i:
                    continue
                a_i = self.vault.agents[i]
                a_j = self.vault.agents[j]
                inter = self.vault.interactions.loc[a_i, a_j]
                if inter == 0:
                    continue
                ref_i = self.vault.library[a_i]["reference"]
                ref_j = self.vault.library[a_j]["reference"]
                rel_i = doses[i] / (ref_i + 1e-8)
                rel_j = doses[j] / (ref_j + 1e-8)
                synergy += inter * np.sqrt(rel_i * rel_j) * 3.0

        base = 100.0
        total = (
            base
            + w["implant"]  * impl
            + w["growth"]   * growth
            + w["neuro"]    * neuro
            + w["vascular"] * vasc
            + w["program"]  * prog
            + synergy
            - tox_total
        )
        return total

# --------------------------- Monte Carlo Wrapper ----------------------
class MonteCarloStageEvaluator:
    """
    For a given dose vector, runs N_sim Monte Carlo draws with:
      - physiological noise
      - response variability
    Returns mean score and std deviation.
    """

    def __init__(self, phys_engine, N_sim=64, noise_level=0.05):
        self.phys = phys_engine
        self.N_sim = N_sim
        self.noise_level = noise_level

    def evaluate(self, doses, stage, age):
        scores = []
        base_score = self.phys.score(doses, stage, age)
        for _ in range(self.N_sim):
            # multiplicative noise to emulate inter-person variability
            eps = np.random.normal(0.0, self.noise_level)
            noisy = base_score * (1.0 + eps)
            scores.append(noisy)
        scores = np.array(scores)
        return scores.mean(), scores.std()

# --------------------------- Optimizer (Outer) ------------------------
class StageOptimizer:
    def __init__(self, vault, mc_evaluator, risk_aversion=0.3):
        self.vault = vault
        self.mc = mc_evaluator
        self.risk_aversion = risk_aversion  # lambda in the objective

    def optimize(self, stage, age, maxiter=40, popsize=16):
        n = self.vault.n_agents

        def objective(x):
            mu, sigma = self.mc.evaluate(x, stage, age)
            # risk-adjusted loss: -mean + lambda * std
            return -(mu - self.risk_aversion * sigma)

        bounds = []
        for agent in self.vault.agents:
            ref = self.vault.library[agent]["reference"]
            high = ref * 10.0 if ref > 0 else 10.0  # wide search; simulation punishes bad regions
            bounds.append((0.0, high))

        result = differential_evolution(
            objective,
            bounds,
            strategy="best1bin",
            maxiter=maxiter,
            popsize=popsize,
            mutation=(0.5, 1.0),
            recombination=0.7,
            seed=2041,
            disp=False,
        )

        best_doses = result.x
        mu, sigma = self.mc.evaluate(best_doses, stage, age)
        return best_doses, mu, sigma

# --------------------------- Orchestrator -----------------------------
class MultiStageNatalityOrchestrator:
    def __init__(self):
        self.vault = NatalityVault()
        self.phys = StagePhysiologyEngine(self.vault)
        self.mc = MonteCarloStageEvaluator(self.phys, N_sim=64, noise_level=0.05)
        self.optimizer = StageOptimizer(self.vault, self.mc, risk_aversion=0.05)

    def run_for_age_and_stage(self, age, stage):
        doses, mu, sigma = self.optimizer.optimize(stage, age)
        return {
            "age": age,
            "stage": stage,
            "mean_score": mu,
            "score_std": sigma,
            "doses": doses,
        }

    def run_full_grid(self):
        results = {}
        for age in AGE_BANDS:
            results[age] = {}
            for stage in STAGES:
                results[age][stage] = self.run_for_age_and_stage(age, stage)
        return results

# --------------------------- ENTRYPOINT -------------------------------
if __name__ == "__main__":
    orchestrator = MultiStageNatalityOrchestrator()
    all_results = orchestrator.run_full_grid()
    # all_results now contains, for each age & stage:
    #   - mean_score based on repeated simulations
    #   - score_std (risk)
    #   - mathematically derived doses (no static "cap" logic)

In [ ]:
# =====================================================================
# CELL 2: PHARMACOMETRICS + REGIMEN GENERATION FOR ALL AGES × STAGES
# =====================================================================
# Prerequisites (from Cell 1):
#   - orchestrator = MultiStageNatalityOrchestrator()
#   - all_results = orchestrator.run_full_grid()
#   - orchestrator.vault (NatalityVault instance)
# =====================================================================

import math

class PharmacometricsEngine:
    """
    Simple first-order PK/PD engine (inspired by your health.ipynb model).
    Converts continuous 'daily-equivalent' dose into:
      - per-administration dose (mg / IU / units)
      - interval (qXh / weekly / monthly) per agent.
    """

    def __init__(self, patient_weight_kg=65.0):
        self.weight = patient_weight_kg

        # PK database: half-life (h), Vd (L/kg), target Css (mg/L), F (bioavailability)
        self.pk_db = {
            # Real-ish or literature-inspired ballpark values (not clinical prescriptions)
            "Metformin (XR)":   {"t_half": 7.0,   "Vd": 1.0,  "C_target": 1.0,  "F": 0.5},
            "Low-Dose Aspirin": {"t_half": 3.2,   "Vd": 0.16, "C_target": 0.02, "F": 0.5},
            "Antihypertensive": {"t_half": 10.0,  "Vd": 2.0,  "C_target": 0.01, "F": 0.3},
            "Insulin (Basal)":  {"t_half": 8.0,   "Vd": 0.1,  "C_target": 0.5,  "F": 1.0},
            "NMN":              {"t_half": 8.0,   "Vd": 0.6,  "C_target": 2.0,  "F": 0.3},
            "Resveratrol":      {"t_half": 9.0,   "Vd": 1.5,  "C_target": 1.0,  "F": 0.3},
            "Spermidine":       {"t_half": 10.0,  "Vd": 0.5,  "C_target": 0.5,  "F": 0.5},
            "PQQ":              {"t_half": 4.0,   "Vd": 0.7,  "C_target": 0.2,  "F": 0.6},
            # Nutrients with daily oral dosing
            "Methylfolate (B9)": {"t_half": 6.0,  "Vd": 0.3,  "C_target": 0.02, "F": 0.7},
            "Iron Bisglycinate": {"t_half": 12.0, "Vd": 0.2,  "C_target": 0.1,  "F": 0.25},
            "Omega-3 (DHA/EPA)": {"t_half": 60.0, "Vd": 3.0,  "C_target": 0.5,  "F": 0.9},
            "Choline (CDP)":     {"t_half": 8.0,  "Vd": 0.7,  "C_target": 0.3,  "F": 0.8},
            "Magnesium Threonate": {"t_half": 10.0, "Vd": 0.3,"C_target": 0.5,  "F": 0.4},
            "Iodine":            {"t_half": 24.0, "Vd": 0.2,  "C_target": 0.05, "F": 1.0},
            "Calcium":           {"t_half": 5.0,  "Vd": 0.3,  "C_target": 0.4,  "F": 0.3},
            "Multi-Trace (Zn/Se)": {"t_half": 24.0,"Vd": 0.2, "C_target": 0.05, "F": 0.4},
            # Experimental biologics as “bolus-like” with long intervals (modeled simply)
            "BPC-157":           {"t_half": 6.0,   "Vd": 0.1, "C_target": 0.001,"F": 0.9},
            "AAV-BDNF":          {"t_half": 24*90, "Vd": 0.05,"C_target": 0.0001,"F": 1.0},
            "Exosomes (MSC)":    {"t_half": 24*7,  "Vd": 0.05,"C_target": 0.0005,"F": 1.0},
        }

    def _maintenance_dose_and_interval(self, agent, daily_equiv):
        """
        daily_equiv: optimized per-day equivalent (mg/day, IU/day, units/day).
        Returns (dose_per_admin, interval_hours, label_string).
        """
        # For agents without PK data, just return daily split
        if agent not in self.pk_db:
            # Daily oral dosing by default
            return daily_equiv, 24.0, "q24h"

        pk = self.pk_db[agent]
        t_half = pk["t_half"]
        Vd = pk["Vd"] * self.weight  # L
        C_target = pk["C_target"]
        F = pk["F"]

        # Clearance CL ≈ (0.693 * Vd) / t_half
        CL = 0.693 * Vd / max(t_half, 1e-3)  # L/h

        # Choose dosing interval tau based on half-life
        if t_half <= 6:
            tau = 12.0   # BID
        elif t_half <= 24:
            tau = 24.0   # q24h
        elif t_half <= 24 * 7:
            tau = 24.0 * 7.0   # weekly
        else:
            tau = 24.0 * 30.0  # monthly

        # Required maintenance dose for Css ~ C_target:
        # Dose = (Css * CL * tau) / F
        dose_pk = (C_target * CL * tau) / max(F, 1e-3)

        # Also tie to the optimizer's daily_equiv by minimizing discrepancy
        dose_from_daily = daily_equiv * (tau / 24.0)

        # Blend both: weighted geometric mean to keep within both logics
        dose = math.sqrt(max(dose_pk, 1e-6) * max(dose_from_daily, 1e-6))

        # Interval label
        if tau == 12.0:
            label = "q12h"
        elif tau == 24.0:
            label = "q24h"
        elif tau == 24.0 * 7.0:
            label = "q7d"
        else:
            label = "q30d"

        return dose, tau, label

    def generate_regimen_for_vector(self, dose_vector, vault, stage, age):
        """
        dose_vector: optimized continuous doses from GENESIS (mg/day / IU/day / units/day).
        Returns a DataFrame with regimen for this age × stage.
        """
        records = []
        for i, agent in enumerate(vault.agents):
            daily = dose_vector[i]
            if daily <= 0.01:
                continue  # essentially not used

            # For IU-like: treat Vitamin D3 as IU/day
            unit = "mg"
            if agent == "Vitamin D3":
                unit = "IU"
            elif agent in ["AAV-BDNF", "Exosomes (MSC)"]:
                unit = "Units"

            dose_per_admin, tau_h, label = self._maintenance_dose_and_interval(agent, daily)

            records.append({
                "Age": age,
                "Stage": stage,
                "Agent": agent,
                "Daily_Equivalent": round(daily, 3),
                "Dose_per_Admin": round(dose_per_admin, 3),
                "Unit": unit,
                "Interval_Hours": tau_h,
                "Interval_Label": label,
            })

        return pd.DataFrame(records)


def build_full_regimen_table(all_results, orchestrator, weight_kg=65.0):
    """
    Takes all_results (from GENESIS v3.0) and returns one big regimen table
    across ages and stages.
    """
    vault = orchestrator.vault
    pk_engine = PharmacometricsEngine(patient_weight_kg=weight_kg)

    all_tables = []
    for age, stages_dict in all_results.items():
        for stage, res in stages_dict.items():
            vec = res["doses"]
            df_reg = pk_engine.generate_regimen_for_vector(vec, vault, stage, age)
            all_tables.append(df_reg)

    if not all_tables:
        return pd.DataFrame()

    full = pd.concat(all_tables, axis=0, ignore_index=True)
    # Sort for readability, but still fully programmatic
    full = full.sort_values(["Age", "Stage", "Agent"]).reset_index(drop=True)
    return full


# =====================================================================
# EXECUTE REGIMEN GENERATION
# =====================================================================

if "all_results" not in globals():
    # Safety: re-run optimizer if needed
    orchestrator = MultiStageNatalityOrchestrator()
    all_results = orchestrator.run_full_grid()

regimen_table = build_full_regimen_table(all_results, orchestrator, weight_kg=65.0)

# At this point, regimen_table is a complete, math-derived schedule for:
#   - Age 20, 30, 40, 50
#   - Stages: preconception, fertilization_window, implantation,
#             trimester_1, trimester_2, trimester_3, postpartum
# You can inspect it, filter it by age/stage, or export to CSV.
# Example (uncomment if you want to *see* it):
# display(regimen_table.head(50))
# regimen_table.to_csv("genesis_natal_regimen_all_ages_stages.csv", index=False)

In [ ]:
# =====================================================================
# CELL 3: HELPER QUERIES FOR AGE × STAGE REGIMENS
# =====================================================================
# Prereqs:
#   - orchestrator, all_results from Cell 1 (GENESIS v3.0)
#   - regimen_table from Cell 2 (PK/PD engine)
# =====================================================================

import pandas as pd

if "regimen_table" not in globals():
    raise RuntimeError("regimen_table is not defined. Run Cell 2 first.")

# Ensure types are clean
regimen_table = regimen_table.copy()
regimen_table["Age"] = regimen_table["Age"].astype(int)
regimen_table["Stage"] = regimen_table["Stage"].astype(str)

STAGES = [
    "preconception",
    "fertilization_window",
    "implantation",
    "trimester_1",
    "trimester_2",
    "trimester_3",
    "postpartum",
]

AGE_BANDS = [20, 30, 40, 50]


def get_regimen(age, stage):
    """
    Return a DataFrame of the optimized regimen for a single age × stage.
    """
    mask = (regimen_table["Age"] == age) & (regimen_table["Stage"] == stage)
    df = regimen_table.loc[mask].sort_values("Agent").reset_index(drop=True)
    return df


def get_age_protocol(age):
    """
    Return the full pregnancy-life-cycle regimen for a given age
    (all stages stacked).
    """
    mask = regimen_table["Age"] == age
    df = regimen_table.loc[mask].sort_values(["Stage", "Agent"]).reset_index(drop=True)
    return df


def get_stage_protocol(stage):
    """
    Return the regimens for all ages at a single stage.
    """
    mask = regimen_table["Stage"] == stage
    df = regimen_table.loc[mask].sort_values(["Age", "Agent"]).reset_index(drop=True)
    return df


# OPTIONAL: convenience dictionaries (no printing, just structures)
regimens_by_age_stage = {
    age: {stage: get_regimen(age, stage) for stage in STAGES}
    for age in AGE_BANDS
}

# Example usage in notebook (uncomment to inspect):
# display(get_regimen(20, "preconception"))
# display(get_age_protocol(30))
# display(get_stage_protocol("trimester_2"))

# You can also export everything if desired:
# regimen_table.to_csv("genesis_regimens_all_ages_stages.csv", index=False)

In [ ]:
# =====================================================================
# CELL 4: HUMAN-READABLE REPORTS & EXPORTS
# =====================================================================
# Prereqs:
#   - orchestrator, all_results  (Cell 1)
#   - regimen_table              (Cell 2)
#   - regimens_by_age_stage      (Cell 3)
# =====================================================================

import pandas as pd

if "all_results" not in globals():
    raise RuntimeError("all_results not found. Run Cell 1.")
if "regimen_table" not in globals():
    raise RuntimeError("regimen_table not found. Run Cell 2.")
if "regimens_by_age_stage" not in globals():
    raise RuntimeError("regimens_by_age_stage not found. Run Cell 3.")

STAGES = [
    "preconception",
    "fertilization_window",
    "implantation",
    "trimester_1",
    "trimester_2",
    "trimester_3",
    "postpartum",
]
AGE_BANDS = [20, 30, 40, 50]

def summarize_age(age):
    """
    Build a concise summary table for one age across all stages.
    Includes:
      - Stage
      - Mean optimality score
      - Score std (risk)
      - Number of active agents
    """
    rows = []
    for stage in STAGES:
        res = all_results[age][stage]
        doses = res["doses"]
        n_active = int((doses > 0.01).sum())
        rows.append({
            "Age": age,
            "Stage": stage,
            "Mean_Score": round(res["mean_score"], 2),
            "Score_Std": round(res["score_std"], 2),
            "Active_Agents": n_active,
        })
    return pd.DataFrame(rows)


def summarize_all_ages():
    frames = [summarize_age(age) for age in AGE_BANDS]
    return pd.concat(frames, axis=0, ignore_index=True)


def get_full_protocol_for_age(age):
    """
    Returns a joined view:
      summary per stage + detailed regimen rows per stage.
    You can filter or pretty-print this as needed in the notebook.
    """
    summary = summarize_age(age)
    reg = get_age_protocol(age)  # from Cell 3

    # Just return both; let the notebook user decide how to display
    return summary, reg


# ---------------------------------------------------------------------
# EXAMPLE: build global summary and (optionally) export
# ---------------------------------------------------------------------

global_summary = summarize_all_ages()

# If you want to *see* it in the notebook, uncomment:
# display(global_summary)

# You can also export the full regimen + summary:
# regimen_table.to_csv("genesis_regimens_all_ages_stages.csv", index=False)
# global_summary.to_csv("genesis_stage_scores_all_ages.csv", index=False)

# Example: get detailed protocol for a specific age (e.g., 30-year-old)
# summary_30, protocol_30 = get_full_protocol_for_age(30)
# display(summary_30)
# display(protocol_30)

In [ ]:
# =====================================================================
# CELL 5: DEMO – PRINT OPTIMAL PROTOCOLS FOR ALL AGES
# =====================================================================
# Prereqs:
#   - global_summary, get_age_protocol, get_regimen (Cells 3 & 4)
#   - all_results, regimen_table
# =====================================================================

import pandas as pd

if "global_summary" not in globals():
    raise RuntimeError("global_summary not defined. Run Cells 1–4 first.")

def print_protocol_for_age(age):
    print("=" * 100)
    print(f"OPTIMIZED NATAL PROTOCOL — AGE {age}")
    print("=" * 100)

    summary, protocol = get_full_protocol_for_age(age)

    # 1) Stage scores
    print("\n[STAGE SCORES]")
    print(summary.to_string(index=False))

    # 2) Detailed regimen per stage
    for stage in STAGES:
        print("\n" + "-" * 100)
        print(f"STAGE: {stage}")
        print("-" * 100)
        df_stage = get_regimen(age, stage)
        if df_stage.empty:
            print("  (no active agents)")
            continue
        # Compact columns
        cols = ["Agent", "Dose_per_Admin", "Unit", "Interval_Label", "Daily_Equivalent"]
        print(df_stage[cols].to_string(index=False))

# ---------------------------------------------------------------------
# PRINT PROTOCOLS FOR ALL AGE BANDS
# ---------------------------------------------------------------------

for age in AGE_BANDS:
    print_protocol_for_age(age)
    print("\n\n")